# 1. Purpose and scope

This notebook builds the reusable Phase 2 data foundation layer for signal expansion. Heavy logic lives in `src/` modules so later notebooks can reuse the same universe, data loading, cleaning, validation, and storage steps.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import sqlite3

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# 2. Imports and paths


In [2]:
from src.run_config import ensure_phase2_nb01_directories, make_run_id, make_run_timestamp
from src.universe import (
    DYNAMIC_TOP300_LIQUIDITY_MODE,
    DYNAMIC_TOP300_LIQUIDITY_VERSION,
    RAW_TICKER_POOL_MODE,
    RAW_TICKER_POOL_VERSION,
    UNIVERSE_LIMITATION_NOTE,
    apply_universe_mask_to_panels,
    build_dynamic_liquidity_membership_table,
    build_dynamic_liquidity_universe_mask,
    get_benchmark_tickers,
    get_phase2_all_tickers,
    get_phase2_stock_universe,
    get_phase2_universe_metadata,
    get_raw_ticker_pool_metadata,
)
from src.data_loader import download_ohlcv_data
from src.data_prep import (
    align_panels,
    basic_data_quality_checks,
    build_ohlcv_panels,
    build_ticker_health_reports,
    enforce_panel_contract,
    missingness_summary,
)
from src.storage import (
    get_phase2_nb01_table_names,
    log_phase2_data_run,
    save_dataframe_csv,
    save_dataframe_parquet,
    write_canonical_and_history_tables,
)

paths = ensure_phase2_nb01_directories()
output_dir = paths['output_dir']
sqlite_db_path = paths['sqlite_db_path']

print(f'Project root: {paths["project_root"]}')
print(f'Output directory: {output_dir}')
print(f'SQLite database: {sqlite_db_path}')


Project root: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model
Output directory: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/data/processed/phase2/nb01_data_foundation
SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db


# 3. Universe config


In [3]:
UNIVERSE_MODE = 'dynamic_top300_liquidity'
UNIVERSE_TOP_N = 300
UNIVERSE_ADV_WINDOW = 20
UNIVERSE_MIN_PRICE = 5.0
UNIVERSE_MIN_VALID_OBS = 15
UNIVERSE_SHIFT_MEMBERSHIP = True

print(f'Universe mode: {UNIVERSE_MODE}')
print(f'Dynamic universe top_n: {UNIVERSE_TOP_N}')
print(f'Dynamic universe ADV window: {UNIVERSE_ADV_WINDOW}')
print(f'Dynamic universe membership shifted: {UNIVERSE_SHIFT_MEMBERSHIP}')


Universe mode: dynamic_top300_liquidity
Dynamic universe top_n: 300
Dynamic universe ADV window: 20
Dynamic universe membership shifted: True


# 4. Create run_id


In [4]:
run_id = make_run_id()
run_timestamp = make_run_timestamp()
print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')


run_id: phase2_nb01_20260507_223029
run_timestamp: 2026-05-07 22:30:29


# 5. Define/load universe and benchmark


In [5]:
stock_universe = get_phase2_stock_universe(mode=UNIVERSE_MODE)
benchmark_tickers = get_benchmark_tickers()
all_tickers = get_phase2_all_tickers(mode=UNIVERSE_MODE, include_benchmarks=True)
universe_metadata = get_phase2_universe_metadata(mode=UNIVERSE_MODE, include_benchmarks=True).copy()
raw_ticker_pool_metadata = (
    get_raw_ticker_pool_metadata()
    if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE
    else pd.DataFrame()
)
universe_metadata['run_id'] = run_id
universe_metadata['timestamp_frozen'] = run_timestamp

universe_name = ', '.join(sorted(universe_metadata['universe_name'].dropna().unique()))
universe_version = ', '.join(sorted(universe_metadata['universe_version'].dropna().unique()))

print(f'Selected universe mode: {UNIVERSE_MODE}')
print(f'Universe version: {universe_version}')
print(f'Raw ticker pool mode: {RAW_TICKER_POOL_MODE if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else "not_applicable"}')
print(f'Raw ticker pool version: {RAW_TICKER_POOL_VERSION if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else "not_applicable"}')
print(f'Raw ticker pool size: {len(stock_universe)}')
print(f'Benchmark tickers: {benchmark_tickers}')
print(f'Total tickers requested: {len(all_tickers)}')

if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE:
    print(f'Universe limitation note: {UNIVERSE_LIMITATION_NOTE}')
display(universe_metadata.head())
if not raw_ticker_pool_metadata.empty:
    display(raw_ticker_pool_metadata.head())


Selected universe mode: dynamic_top300_liquidity
Universe version: dynamic_top300_from_current_large_liquid_pool_v1
Raw ticker pool mode: current_large_liquid_pool_v1
Raw ticker pool version: current_large_liquid_pool_v1
Raw ticker pool size: 488
Benchmark tickers: ['SPY']
Total tickers requested: 489
Universe limitation note: Dynamic top-300 liquidity selection is applied to a current large/liquid ticker pool. This is useful for engineering robustness testing, but not fully survivorship-free. A fully survivorship-free test requires historical constituent membership or a point-in-time security master.


,ticker,source,universe_name,universe_version,active,notes,run_id,timestamp_frozen
0,A,phase2_universe_module_dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool,dynamic_top300_from_current_large_liquid_pool_v1,True,Dynamic top-300 liquidity selection is applied...,phase2_nb01_20260507_223029,2026-05-07 22:30:29
1,AAPL,phase2_universe_module_dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool,dynamic_top300_from_current_large_liquid_pool_v1,True,Dynamic top-300 liquidity selection is applied...,phase2_nb01_20260507_223029,2026-05-07 22:30:29
2,ABBV,phase2_universe_module_dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool,dynamic_top300_from_current_large_liquid_pool_v1,True,Dynamic top-300 liquidity selection is applied...,phase2_nb01_20260507_223029,2026-05-07 22:30:29
3,ABNB,phase2_universe_module_dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool,dynamic_top300_from_current_large_liquid_pool_v1,True,Dynamic top-300 liquidity selection is applied...,phase2_nb01_20260507_223029,2026-05-07 22:30:29
4,ABT,phase2_universe_module_dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool,dynamic_top300_from_current_large_liquid_pool_v1,True,Dynamic top-300 liquidity selection is applied...,phase2_nb01_20260507_223029,2026-05-07 22:30:29


,ticker,source_pool,inclusion_reason,raw_pool_version,limitation_note
0,A,current_large_liquid_pool_v1,Current large/liquid US equity common-stock en...,current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
1,AAPL,current_large_liquid_pool_v1,Current large/liquid US equity common-stock en...,current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
2,ABBV,current_large_liquid_pool_v1,Current large/liquid US equity common-stock en...,current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
3,ABNB,current_large_liquid_pool_v1,Current large/liquid US equity common-stock en...,current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
4,ABT,current_large_liquid_pool_v1,Current large/liquid US equity common-stock en...,current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...


# 6. Download/load OHLCV data using src/data_loader.py


In [6]:
start_date = '2018-01-01'
end_date = None
auto_adjust = True

raw_ohlcv, download_status = download_ohlcv_data(
    tickers=all_tickers,
    start_date=start_date,
    end_date=end_date,
    auto_adjust=auto_adjust,
)

print(raw_ohlcv.shape)
display(download_status)

$MMC: possibly delisted; no timezone found


$JNPR: possibly delisted; no timezone found


$DFS: possibly delisted; no timezone found


$HES: possibly delisted; no timezone found


$WBA: possibly delisted; no timezone found


...

$ANSS: possibly delisted; no timezone found



11 Failed downloads:


['MMC', 'JNPR', 'DFS', 'HES', 'WBA', 'DAY', 'FI', 'CTLT', 'IPG', 'K', 'ANSS']: possibly delisted; no timezone found


(2098, 2456)


,ticker,status,non_null_close_rows,first_valid_date,last_valid_date
0,A,ok,2098,2018-01-02,2026-05-07
1,AAPL,ok,2098,2018-01-02,2026-05-07
2,ABBV,ok,2098,2018-01-02,2026-05-07
3,ABNB,ok,1357,2020-12-10,2026-05-07
4,ABT,ok,2098,2018-01-02,2026-05-07
...,...,...,...,...,...
484,YUM,ok,2098,2018-01-02,2026-05-07
485,ZBH,ok,2098,2018-01-02,2026-05-07
486,ZBRA,ok,2098,2018-01-02,2026-05-07
487,ZTS,ok,2098,2018-01-02,2026-05-07


# 7. Clean and align OHLCV panels using src/data_prep.py


In [7]:
ohlcv_panels = build_ohlcv_panels(raw_ohlcv)
aligned_panels = align_panels(ohlcv_panels)
aligned_panels = {name: enforce_panel_contract(panel, name) for name, panel in aligned_panels.items()}
ticker_health_reports = build_ticker_health_reports(aligned_panels)

open_prices = aligned_panels['open']
high_prices = aligned_panels['high']
low_prices = aligned_panels['low']
close_prices = aligned_panels['close']
volume = aligned_panels['volume']

print('Aligned panel shapes:')
for name, panel in aligned_panels.items():
    print(f'  {name}: {panel.shape}')


Aligned panel shapes:
  open: (2098, 478)
  high: (2098, 478)
  low: (2098, 478)
  close: (2098, 478)
  volume: (2098, 478)


# 7B. Build and apply dynamic top-300 liquidity universe


In [8]:
pre_universe_panel_shapes = {name: panel.shape for name, panel in aligned_panels.items()}
dynamic_universe_membership = pd.DataFrame()
dynamic_universe_diagnostics = pd.DataFrame()
universe_mask_for_trading = pd.DataFrame(index=close_prices.index, columns=close_prices.columns, data=True)
universe_tables_written = []

if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE:
    stock_columns = [ticker for ticker in stock_universe if ticker in close_prices.columns]
    stock_close_prices = close_prices.loc[:, stock_columns].copy()
    stock_volume = volume.loc[:, stock_columns].copy()
    universe_mask_for_trading, dynamic_universe_diagnostics = build_dynamic_liquidity_universe_mask(
        close_prices=stock_close_prices,
        volume=stock_volume,
        top_n=UNIVERSE_TOP_N,
        adv_window=UNIVERSE_ADV_WINDOW,
        min_price=UNIVERSE_MIN_PRICE,
        min_valid_obs=UNIVERSE_MIN_VALID_OBS,
        shift_membership=UNIVERSE_SHIFT_MEMBERSHIP,
    )
    dynamic_universe_membership = build_dynamic_liquidity_membership_table(
        close_prices=stock_close_prices,
        volume=stock_volume,
        universe_mask=universe_mask_for_trading,
        top_n=UNIVERSE_TOP_N,
        adv_window=UNIVERSE_ADV_WINDOW,
        min_valid_obs=UNIVERSE_MIN_VALID_OBS,
        shift_membership=UNIVERSE_SHIFT_MEMBERSHIP,
    )
    aligned_panels = apply_universe_mask_to_panels(
        panels=aligned_panels,
        universe_mask=universe_mask_for_trading,
        benchmark_tickers=benchmark_tickers,
    )
    aligned_panels = {name: enforce_panel_contract(panel, name) for name, panel in aligned_panels.items()}
    ticker_health_reports = build_ticker_health_reports(aligned_panels)

    open_prices = aligned_panels['open']
    high_prices = aligned_panels['high']
    low_prices = aligned_panels['low']
    close_prices = aligned_panels['close']
    volume = aligned_panels['volume']

post_universe_panel_shapes = {name: panel.shape for name, panel in aligned_panels.items()}
selected_count_summary = (
    dynamic_universe_diagnostics['n_selected_tickers'].describe().to_frame('n_selected_tickers')
    if not dynamic_universe_diagnostics.empty
    else pd.DataFrame()
)

print(f'Universe mode/version: {UNIVERSE_MODE} / {universe_version}')
print(f'Raw ticker pool size: {len(stock_universe)}')
print(f'Successfully downloaded stock tickers: {len(stock_columns) if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else len(stock_universe)}')
if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE:
    print(UNIVERSE_LIMITATION_NOTE)
print(f'Mask shifted by one trading day: {UNIVERSE_SHIFT_MEMBERSHIP if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else "not_applicable"}')
print('Before/after panel shapes')
display(pd.DataFrame([
    {'panel': name, 'before_shape': pre_universe_panel_shapes[name], 'after_shape': post_universe_panel_shapes[name]}
    for name in sorted(post_universe_panel_shapes)
]))
if not selected_count_summary.empty:
    display(selected_count_summary)
if not dynamic_universe_diagnostics.empty:
    display(dynamic_universe_diagnostics.head())
if not dynamic_universe_membership.empty:
    display(dynamic_universe_membership.head())


Universe mode/version: dynamic_top300_liquidity / dynamic_top300_from_current_large_liquid_pool_v1
Raw ticker pool size: 488
Successfully downloaded stock tickers: 477
Dynamic top-300 liquidity selection is applied to a current large/liquid ticker pool. This is useful for engineering robustness testing, but not fully survivorship-free. A fully survivorship-free test requires historical constituent membership or a point-in-time security master.
Mask shifted by one trading day: True
Before/after panel shapes


,panel,before_shape,after_shape
0,close,"(2098, 478)","(2098, 478)"
1,high,"(2098, 478)","(2098, 478)"
2,low,"(2098, 478)","(2098, 478)"
3,open,"(2098, 478)","(2098, 478)"
4,volume,"(2098, 478)","(2098, 478)"


,n_selected_tickers
count,2098.000000
mean,297.855100
std,25.281891
min,0.000000
25%,300.000000
50%,300.000000
75%,300.000000
max,300.000000


,Date,n_available_tickers,n_liquid_eligible_tickers,n_selected_tickers,median_adv20,min_selected_adv20,max_selected_adv20,pct_missing_close,pct_missing_volume,shift_membership,adv_window,top_n,min_price,min_valid_obs,universe_mode,universe_version,limitation_note
0,2018-01-02,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
1,2018-01-03,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
2,2018-01-04,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
3,2018-01-05,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
4,2018-01-08,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...


,Date,ticker,in_universe,adv20,close,volume,universe_rank,universe_mode,universe_version
0,2018-01-24,A,True,1.230661e+08,69.142120,1754300.0,259.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
1,2018-01-24,AAPL,True,4.376121e+09,40.762760,204420400.0,2.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
2,2018-01-24,ABBV,True,3.584067e+08,74.263863,4436500.0,82.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
3,2018-01-24,ABT,True,3.245469e+08,53.276802,11704900.0,90.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
4,2018-01-24,ACN,True,2.993587e+08,141.514877,1951900.0,104.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1


# 8. Build benchmark prices


In [9]:
benchmark_close_prices = close_prices.loc[:, benchmark_tickers].copy()
display(benchmark_close_prices.head())

Ticker,SPY
Date,
2018-01-02,236.562164
2018-01-03,238.058411
2018-01-04,239.061859
2018-01-05,240.654953
2018-01-08,241.095078


# 9. Run data quality checks


In [10]:
close_missingness = missingness_summary(close_prices)
quality_checks = basic_data_quality_checks(aligned_panels, expected_tickers=all_tickers)
ticker_health_close = ticker_health_reports['close']
ticker_health_open = ticker_health_reports['open']
ticker_health_high = ticker_health_reports['high']
ticker_health_low = ticker_health_reports['low']
ticker_health_volume = ticker_health_reports['volume']

display(quality_checks)
display(close_missingness.head(10))
display(ticker_health_close.head(10))


,panel,row_count,ticker_count,index_is_datetime,index_is_sorted,duplicate_dates,duplicate_tickers,inf_count,all_empty_tickers,null_column_labels,numeric_columns_only,missing_expected_tickers,passed
0,open,2098,478,True,True,0,0,0,16,0,True,"ANSS, CTLT, DAY, DFS, FI, HES, IPG, JNPR, K, M...",False
1,high,2098,478,True,True,0,0,0,16,0,True,"ANSS, CTLT, DAY, DFS, FI, HES, IPG, JNPR, K, M...",False
2,low,2098,478,True,True,0,0,0,16,0,True,"ANSS, CTLT, DAY, DFS, FI, HES, IPG, JNPR, K, M...",False
3,close,2098,478,True,True,0,0,0,16,0,True,"ANSS, CTLT, DAY, DFS, FI, HES, IPG, JNPR, K, M...",False
4,volume,2098,478,True,True,0,0,0,16,0,True,"ANSS, CTLT, DAY, DFS, FI, HES, IPG, JNPR, K, M...",False


,ticker,missing_count,missing_pct,non_null_count
0,ALLE,2098,1.0,0
1,AVY,2098,1.0,0
2,BEN,2098,1.0,0
3,CINF,2098,1.0,0
4,FRT,2098,1.0,0
5,IVZ,2098,1.0,0
6,L,2098,1.0,0
7,LKQ,2098,1.0,0
8,LNT,2098,1.0,0
9,MAA,2098,1.0,0


,panel_name,ticker,n_obs,missing_count,missing_pct,first_valid_date,last_valid_date
0,close,A,1809,289,0.137750,2018-01-24,2026-03-27
1,close,AAPL,2083,15,0.007150,2018-01-24,2026-05-07
2,close,ABBV,2083,15,0.007150,2018-01-24,2026-05-07
3,close,ABNB,1342,756,0.360343,2021-01-04,2026-05-07
4,close,ABT,2083,15,0.007150,2018-01-24,2026-05-07
5,close,ACGL,94,2004,0.955195,2022-11-01,2025-08-05
6,close,ACN,2083,15,0.007150,2018-01-24,2026-05-07
7,close,ADBE,2083,15,0.007150,2018-01-24,2026-05-07
8,close,ADI,2083,15,0.007150,2018-01-24,2026-05-07
9,close,ADM,1173,925,0.440896,2018-01-24,2026-04-20


# 10. Save canonical outputs to processed/phase2/nb01_data_foundation/


In [11]:
failed_tickers = download_status.loc[download_status['status'] != 'ok', 'ticker'].tolist()
data_contract_summary = pd.DataFrame([
    {
        'run_id': run_id,
        'run_timestamp': run_timestamp,
        'universe_mode': UNIVERSE_MODE,
        'universe_version': universe_version,
        'universe_limitation_note': UNIVERSE_LIMITATION_NOTE if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else '',
        'raw_ticker_pool_mode': RAW_TICKER_POOL_MODE if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else '',
        'raw_ticker_pool_version': RAW_TICKER_POOL_VERSION if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else '',
        'raw_ticker_pool_size': len(stock_universe),
        'n_tickers': close_prices.shape[1],
        'n_dates': len(close_prices),
        'date_start': close_prices.index.min(),
        'date_end': close_prices.index.max(),
        'average_missing_pct_close': float(close_missingness['missing_pct'].mean()),
        'benchmark_tickers': ', '.join(benchmark_tickers),
        'output_dir': str(output_dir),
        'sqlite_path': str(sqlite_db_path),
    }
])

output_paths = {}
output_paths['universe_metadata_csv'] = save_dataframe_csv(universe_metadata, output_dir / 'universe_metadata.csv', index=False)
if not raw_ticker_pool_metadata.empty:
    output_paths['raw_ticker_pool_csv'] = save_dataframe_csv(raw_ticker_pool_metadata, output_dir / 'raw_ticker_pool.csv', index=False)
output_paths['download_status_csv'] = save_dataframe_csv(download_status, output_dir / 'download_status.csv', index=False)
output_paths['raw_ohlcv_parquet'] = save_dataframe_parquet(raw_ohlcv, output_dir / 'raw_ohlcv.parquet', index=True)
output_paths['open_prices_parquet'] = save_dataframe_parquet(open_prices, output_dir / 'open_prices.parquet', index=True)
output_paths['high_prices_parquet'] = save_dataframe_parquet(high_prices, output_dir / 'high_prices.parquet', index=True)
output_paths['low_prices_parquet'] = save_dataframe_parquet(low_prices, output_dir / 'low_prices.parquet', index=True)
output_paths['close_prices_parquet'] = save_dataframe_parquet(close_prices, output_dir / 'close_prices.parquet', index=True)
output_paths['volume_parquet'] = save_dataframe_parquet(volume, output_dir / 'volume.parquet', index=True)
output_paths['benchmark_close_csv'] = save_dataframe_csv(benchmark_close_prices, output_dir / 'benchmark_close_prices.csv', index=True)
output_paths['close_missingness_csv'] = save_dataframe_csv(close_missingness, output_dir / 'close_missingness_summary.csv', index=False)
output_paths['quality_checks_csv'] = save_dataframe_csv(quality_checks, output_dir / 'data_quality_checks.csv', index=False)
output_paths['ticker_health_close_csv'] = save_dataframe_csv(ticker_health_close, output_dir / 'ticker_health_close.csv', index=False)
output_paths['ticker_health_open_csv'] = save_dataframe_csv(ticker_health_open, output_dir / 'ticker_health_open.csv', index=False)
output_paths['ticker_health_high_csv'] = save_dataframe_csv(ticker_health_high, output_dir / 'ticker_health_high.csv', index=False)
output_paths['ticker_health_low_csv'] = save_dataframe_csv(ticker_health_low, output_dir / 'ticker_health_low.csv', index=False)
output_paths['ticker_health_volume_csv'] = save_dataframe_csv(ticker_health_volume, output_dir / 'ticker_health_volume.csv', index=False)
output_paths['data_contract_summary_csv'] = save_dataframe_csv(data_contract_summary, output_dir / 'data_contract_summary.csv', index=False)
if not dynamic_universe_diagnostics.empty:
    output_paths['dynamic_universe_diagnostics_csv'] = save_dataframe_csv(dynamic_universe_diagnostics, output_dir / 'dynamic_top300_universe_diagnostics.csv', index=False)
if not dynamic_universe_membership.empty:
    output_paths['dynamic_universe_membership_csv'] = save_dataframe_csv(dynamic_universe_membership, output_dir / 'dynamic_top300_universe_membership.csv', index=False)

display(pd.Series({key: str(value) for key, value in output_paths.items()}, name='path'))


universe_metadata_csv               /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
raw_ticker_pool_csv                 /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
download_status_csv                 /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
raw_ohlcv_parquet                   /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
open_prices_parquet                 /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
high_prices_parquet                 /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
low_prices_parquet                  /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
close_prices_parquet                /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
volume_parquet                      /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
benchmark_close_csv                 /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
close_missingness_csv               /Users/AnyiXu_1/Desktop/multi-factor-equity-al...
quality_checks_csv                  /Users/AnyiXu_1/De

# 11. Write canonical tables to SQLite


In [12]:
write_canonical_and_history_tables(
    universe_metadata,
    canonical_table=get_phase2_nb01_table_names('universe_metadata')[0],
    history_table=get_phase2_nb01_table_names('universe_metadata')[1],
    db_path=sqlite_db_path,
    run_id=None,
)

write_canonical_and_history_tables(
    close_prices,
    canonical_table=get_phase2_nb01_table_names('close')[0],
    history_table=get_phase2_nb01_table_names('close')[1],
    db_path=sqlite_db_path,
    run_id=run_id,
)

write_canonical_and_history_tables(
    open_prices,
    canonical_table=get_phase2_nb01_table_names('open')[0],
    history_table=get_phase2_nb01_table_names('open')[1],
    db_path=sqlite_db_path,
    run_id=run_id,
)

write_canonical_and_history_tables(
    high_prices,
    canonical_table=get_phase2_nb01_table_names('high')[0],
    history_table=get_phase2_nb01_table_names('high')[1],
    db_path=sqlite_db_path,
    run_id=run_id,
)

write_canonical_and_history_tables(
    low_prices,
    canonical_table=get_phase2_nb01_table_names('low')[0],
    history_table=get_phase2_nb01_table_names('low')[1],
    db_path=sqlite_db_path,
    run_id=run_id,
)

write_canonical_and_history_tables(
    volume,
    canonical_table=get_phase2_nb01_table_names('volume')[0],
    history_table=get_phase2_nb01_table_names('volume')[1],
    db_path=sqlite_db_path,
    run_id=run_id,
)

write_canonical_and_history_tables(
    benchmark_close_prices,
    canonical_table=get_phase2_nb01_table_names('benchmark')[0],
    history_table=get_phase2_nb01_table_names('benchmark')[1],
    db_path=sqlite_db_path,
    run_id=run_id,
)



if not raw_ticker_pool_metadata.empty:
    write_canonical_and_history_tables(
        raw_ticker_pool_metadata,
        canonical_table=get_phase2_nb01_table_names('raw_ticker_pool')[0],
        history_table=get_phase2_nb01_table_names('raw_ticker_pool')[1],
        db_path=sqlite_db_path,
        run_id=run_id,
    )
    universe_tables_written.extend(get_phase2_nb01_table_names('raw_ticker_pool'))

if not dynamic_universe_membership.empty:
    write_canonical_and_history_tables(
        dynamic_universe_membership,
        canonical_table=get_phase2_nb01_table_names('dynamic_top300_membership')[0],
        history_table=get_phase2_nb01_table_names('dynamic_top300_membership')[1],
        db_path=sqlite_db_path,
        run_id=run_id,
    )
    universe_tables_written.extend(get_phase2_nb01_table_names('dynamic_top300_membership'))

if not dynamic_universe_diagnostics.empty:
    write_canonical_and_history_tables(
        dynamic_universe_diagnostics,
        canonical_table=get_phase2_nb01_table_names('dynamic_top300_diagnostics')[0],
        history_table=get_phase2_nb01_table_names('dynamic_top300_diagnostics')[1],
        db_path=sqlite_db_path,
        run_id=run_id,
    )
    universe_tables_written.extend(get_phase2_nb01_table_names('dynamic_top300_diagnostics'))

log_phase2_data_run(
    run_id=run_id,
    metadata={
        'run_timestamp': run_timestamp,
        'universe_mode': UNIVERSE_MODE,
        'universe_name': universe_name,
        'universe_version': universe_version,
        'benchmark_tickers': ', '.join(benchmark_tickers),
        'n_tickers': int(close_prices.shape[1]),
        'n_dates': int(len(close_prices)),
        'start_date': close_prices.index.min(),
        'end_date': close_prices.index.max(),
        'output_dir': str(output_dir),
        'notes': f'universe_mode={UNIVERSE_MODE}; universe_version={universe_version}; raw_pool_mode={RAW_TICKER_POOL_MODE if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else ""}; raw_pool_size={len(stock_universe)}; limitation_note={UNIVERSE_LIMITATION_NOTE if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else ""}; shift_membership={UNIVERSE_SHIFT_MEMBERSHIP if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else "not_applicable"}; average_missing_pct_close={float(close_missingness["missing_pct"].mean()):.6f}; failed_tickers={failed_tickers}',
    },
    sqlite_path=sqlite_db_path,
)

print(f'Wrote canonical tables to {sqlite_db_path}')


Wrote canonical tables to /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db


# 12. Display final summary: row count, ticker count, date range, missingness summary, output paths


In [13]:
display(data_contract_summary)
display(close_missingness.head(10))
display(pd.DataFrame({'artifact': list(output_paths.keys()), 'path': [str(path) for path in output_paths.values()]}))


,run_id,run_timestamp,universe_mode,universe_version,universe_limitation_note,raw_ticker_pool_mode,raw_ticker_pool_version,raw_ticker_pool_size,n_tickers,n_dates,date_start,date_end,average_missing_pct_close,benchmark_tickers,output_dir,sqlite_path
0,phase2_nb01_20260507_223029,2026-05-07 22:30:29,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...,current_large_liquid_pool_v1,current_large_liquid_pool_v1,488,478,2098,2018-01-02,2026-05-07,0.374781,SPY,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


,ticker,missing_count,missing_pct,non_null_count
0,ALLE,2098,1.0,0
1,AVY,2098,1.0,0
2,BEN,2098,1.0,0
3,CINF,2098,1.0,0
4,FRT,2098,1.0,0
5,IVZ,2098,1.0,0
6,L,2098,1.0,0
7,LKQ,2098,1.0,0
8,LNT,2098,1.0,0
9,MAA,2098,1.0,0


,artifact,path
0,universe_metadata_csv,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,raw_ticker_pool_csv,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,download_status_csv,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,raw_ohlcv_parquet,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,open_prices_parquet,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,high_prices_parquet,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,low_prices_parquet,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
7,close_prices_parquet,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
8,volume_parquet,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
9,benchmark_close_csv,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


# 13. Final run status


In [14]:
final_status = pd.DataFrame([
    {
        'run_id': run_id,
        'universe_mode': UNIVERSE_MODE,
        'universe_version': universe_version,
        'raw_ticker_pool_size': len(stock_universe),
        'downloaded_stock_tickers': len([ticker for ticker in stock_universe if ticker in close_prices.columns]),
        'n_tickers': int(close_prices.shape[1]),
        'date_range': f"{close_prices.index.min().date()} to {close_prices.index.max().date()}",
        'output_dir': str(output_dir),
        'sqlite_db_path': str(sqlite_db_path),
        'failed_tickers': ', '.join(failed_tickers) if failed_tickers else 'None',
        'universe_limitation_note': UNIVERSE_LIMITATION_NOTE if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else '',
        'mask_shifted_by_one_day': UNIVERSE_SHIFT_MEMBERSHIP if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE else 'not_applicable',
    }
])

print(f"run_id: {run_id}")
print(f"selected universe mode: {UNIVERSE_MODE}")
print(f"universe version: {universe_version}")
print(f"raw ticker pool size: {len(stock_universe)}")
print(f"number of tickers successfully downloaded in clean panel: {len([ticker for ticker in stock_universe if ticker in close_prices.columns])}")
print(f"number of tickers in clean panel including benchmark: {close_prices.shape[1]}")
print(f"date range: {close_prices.index.min().date()} to {close_prices.index.max().date()}")
print(f"saved output folder: {output_dir}")
print(f"SQLite database path: {sqlite_db_path}")
print(f"failed tickers from yfinance: {failed_tickers if failed_tickers else ['None']}")
if UNIVERSE_MODE == DYNAMIC_TOP300_LIQUIDITY_MODE:
    print(f"universe limitation note: {UNIVERSE_LIMITATION_NOTE}")
    print(f"SQLite dynamic universe tables written: {universe_tables_written}")
display(final_status)


verification_checks = pd.DataFrame([
    {
        'check_name': 'dynamic_diagnostics_written_or_not_dynamic',
        'passed': bool(UNIVERSE_MODE != DYNAMIC_TOP300_LIQUIDITY_MODE or not dynamic_universe_diagnostics.empty),
    },
    {
        'check_name': 'dynamic_membership_written_or_not_dynamic',
        'passed': bool(UNIVERSE_MODE != DYNAMIC_TOP300_LIQUIDITY_MODE or not dynamic_universe_membership.empty),
    },
    {
        'check_name': 'membership_shift_enabled',
        'passed': bool(UNIVERSE_MODE != DYNAMIC_TOP300_LIQUIDITY_MODE or UNIVERSE_SHIFT_MEMBERSHIP),
    },
    {
        'check_name': 'raw_ticker_pool_size_gt_300',
        'passed': bool(UNIVERSE_MODE != DYNAMIC_TOP300_LIQUIDITY_MODE or len(stock_universe) > 300),
    },
    {
        'check_name': 'dynamic_max_selected_close_to_300',
        'passed': bool(UNIVERSE_MODE != DYNAMIC_TOP300_LIQUIDITY_MODE or dynamic_universe_diagnostics['n_selected_tickers'].max() >= 280),
    },
    {
        'check_name': 'average_selected_above_old_pool',
        'passed': bool(UNIVERSE_MODE != DYNAMIC_TOP300_LIQUIDITY_MODE or dynamic_universe_diagnostics['n_selected_tickers'].mean() > 150),
    },
    {
        'check_name': 'average_selected_close_to_available_eligible',
        'passed': bool(
            UNIVERSE_MODE != DYNAMIC_TOP300_LIQUIDITY_MODE
            or dynamic_universe_diagnostics['n_selected_tickers'].mean()
            <= dynamic_universe_diagnostics['n_liquid_eligible_tickers'].mean()
        ),
    },
])

print('Verification checks')
display(verification_checks)
if not dynamic_universe_diagnostics.empty:
    print('Selected ticker count summary')
    display(dynamic_universe_diagnostics['n_selected_tickers'].describe().to_frame('n_selected_tickers'))
    print(f"avg selected tickers/date: {dynamic_universe_diagnostics['n_selected_tickers'].mean():.2f}")
    print(f"min selected tickers/date: {dynamic_universe_diagnostics['n_selected_tickers'].min()}")
    print(f"max selected tickers/date: {dynamic_universe_diagnostics['n_selected_tickers'].max()}")
    print(f"Latest selected ticker count: {dynamic_universe_diagnostics.sort_values('Date')['n_selected_tickers'].iloc[-1]}")
    print('Diagnostics sample')
    display(dynamic_universe_diagnostics.head())
if not dynamic_universe_membership.empty:
    print('Selected membership sample')
    display(dynamic_universe_membership.head())


run_id: phase2_nb01_20260507_223029
selected universe mode: dynamic_top300_liquidity
universe version: dynamic_top300_from_current_large_liquid_pool_v1
raw ticker pool size: 488
number of tickers successfully downloaded in clean panel: 477
number of tickers in clean panel including benchmark: 478
date range: 2018-01-02 to 2026-05-07
saved output folder: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/data/processed/phase2/nb01_data_foundation
SQLite database path: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
failed tickers from yfinance: ['ANSS', 'CTLT', 'DAY', 'DFS', 'FI', 'HES', 'IPG', 'JNPR', 'K', 'MMC', 'WBA']
universe limitation note: Dynamic top-300 liquidity selection is applied to a current large/liquid ticker pool. This is useful for engineering robustness testing, but not fully survivorship-free. A fully survivorship-free test requires historical constituent membership or a point-in-time security master.
SQLite dynamic universe table

,run_id,universe_mode,universe_version,raw_ticker_pool_size,downloaded_stock_tickers,n_tickers,date_range,output_dir,sqlite_db_path,failed_tickers,universe_limitation_note,mask_shifted_by_one_day
0,phase2_nb01_20260507_223029,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,488,477,478,2018-01-02 to 2026-05-07,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...,"ANSS, CTLT, DAY, DFS, FI, HES, IPG, JNPR, K, M...",Dynamic top-300 liquidity selection is applied...,True


Verification checks


,check_name,passed
0,dynamic_diagnostics_written_or_not_dynamic,True
1,dynamic_membership_written_or_not_dynamic,True
2,membership_shift_enabled,True
3,raw_ticker_pool_size_gt_300,True
4,dynamic_max_selected_close_to_300,True
5,average_selected_above_old_pool,True
6,average_selected_close_to_available_eligible,True


Selected ticker count summary


,n_selected_tickers
count,2098.000000
mean,297.855100
std,25.281891
min,0.000000
25%,300.000000
50%,300.000000
75%,300.000000
max,300.000000


avg selected tickers/date: 297.86
min selected tickers/date: 0
max selected tickers/date: 300
Latest selected ticker count: 300
Diagnostics sample


,Date,n_available_tickers,n_liquid_eligible_tickers,n_selected_tickers,median_adv20,min_selected_adv20,max_selected_adv20,pct_missing_close,pct_missing_volume,shift_membership,adv_window,top_n,min_price,min_valid_obs,universe_mode,universe_version,limitation_note
0,2018-01-02,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
1,2018-01-03,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
2,2018-01-04,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
3,2018-01-05,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...
4,2018-01-08,461,0,0,NaN,NaN,NaN,0.033543,0.033543,True,20,300,5.0,15,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1,Dynamic top-300 liquidity selection is applied...


Selected membership sample


,Date,ticker,in_universe,adv20,close,volume,universe_rank,universe_mode,universe_version
0,2018-01-24,A,True,1.230661e+08,69.142120,1754300.0,259.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
1,2018-01-24,AAPL,True,4.376121e+09,40.762760,204420400.0,2.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
2,2018-01-24,ABBV,True,3.584067e+08,74.263863,4436500.0,82.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
3,2018-01-24,ABT,True,3.245469e+08,53.276802,11704900.0,90.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1
4,2018-01-24,ACN,True,2.993587e+08,141.514877,1951900.0,104.0,dynamic_top300_liquidity,dynamic_top300_from_current_large_liquid_pool_v1


In [15]:
from src.db import connect_db, list_tables, load_price_table

sqlite_tables = list_tables(sqlite_db_path)
table_row_counts = []
with connect_db(sqlite_db_path) as conn:
    for table_name in sqlite_tables['name']:
        quoted_table_name = '"' + table_name.replace('"', '""') + '"'
        row_count = pd.read_sql_query(f'SELECT COUNT(*) AS row_count FROM {quoted_table_name}', conn).loc[0, 'row_count']
        table_row_counts.append({'table_name': table_name, 'row_count': int(row_count)})

close_prices_from_db = load_price_table('clean_close_prices_current', db_path=sqlite_db_path)

display(sqlite_tables)
display(pd.DataFrame(table_row_counts).sort_values('table_name').reset_index(drop=True))
print(f'clean_close_prices_current shape from SQLite: {close_prices_from_db.shape}')
display(close_prices_from_db.head())

,name
0,alpha_best_horizon_current
1,alpha_best_horizon_history
2,alpha_candidates_current
3,alpha_candidates_history
4,alpha_constructed_candidates_current
...,...
246,wfv_window_diagnostics_history
247,wfv_window_results_current
248,wfv_window_results_history
249,wfv_windows_current


,table_name,row_count
0,alpha_best_horizon_current,10
1,alpha_best_horizon_history,40
2,alpha_candidates_current,2108880
3,alpha_candidates_history,8435520
4,alpha_constructed_candidates_current,2108880
...,...,...
246,wfv_window_diagnostics_history,12
247,wfv_window_results_current,60
248,wfv_window_results_history,480
249,wfv_windows_current,4


clean_close_prices_current shape from SQLite: (2098, 478)


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
from src.db import load_price_table, load_ohlcv_panels

close = load_price_table("clean_close_prices_current")

print(close.shape)
print(close.index.name)
print("run_id in columns?", "run_id" in close.columns)

panels = load_ohlcv_panels()
print({k: v.shape for k, v in panels.items()})

(2098, 478)
Date
run_id in columns? False


{'open': (2098, 478), 'high': (2098, 478), 'low': (2098, 478), 'close': (2098, 478), 'volume': (2098, 478)}
